# Flow Configuration Testing

This notebook tests the FlowConfiguration model with different parameter sources and modules.

In [1]:
import sys
sys.path.append('..')

from ped.flow import FlowConfiguration
import json
from pprint import pprint

/opt/miniconda/envs/dspdev/lib/python3.10/site-packages/pyspark/pandas/__init__.py:43: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


In [2]:
from ped.param.sources import ParameterSource

In [3]:
test_config = {
    "metadata": {
        "name": "Risk Assessment Flow",
        "description": "A flow for assessing customer risk using decision tables and inline calculations",
        "author": "Risk Team",
        "license": "MIT",
        "version": "1.0"
    },
    "parameter_sources": {
        "static_config": {
            "type": "static",
            "values": {
                "max_risk_score": 100,
                "default_threshold": 0.7,
                "min_credit_score": 650,
                "max_loan_amount": 500000,
                "interest_rates": {
                    "low_risk": 3.2,
                    "medium_risk": 4.5,
                    "high_risk": 6.8
                }
            }
        },
        "request_params": {
            "type": "inputs",
            "base_key": "parameters",
            "version_key": "version",
            "defaults": {
                "environment": "production",
                "debug_mode": False,
                "max_processing_time": 5000
            }
        }
    },

    "modules": {
        "risk_calculator": {
            "type": "inline",
            "test": "${ped.param:static_config,max_risk_score,test}",
            "test2": "${ped.param:request_params,max_processing_time}",
            "test3": "${oc.env:TEST_ENV_VAR,default_value}"

        }
    }
}

In [4]:
sample_request = {
    "parameters": {
        "environment": "staging",
        "debug_mode": True,
        "max_processing_time": 3000,
        "version": "v1.2"
    },
    "customer_data": {
        "age": 32,
        "income": 75000,
        "credit_score": 720
    }
}
flow_config = FlowConfiguration.model_validate(test_config)

parameterized_modules = await flow_config.get_parameterized_module_config(inputs=sample_request)
parameterized_modules

{'risk_calculator': {'type': 'inline',
  'test': 100,
  'test2': 3000,
  'test3': 'default_value'}}

In [ ]:
test_config = {
    "outputs": ["config_example.a", "config_example.b"],
    "metadata": {
        "name": "Risk Assessment Flow",
        "description": "A flow for assessing customer risk using decision tables and inline calculations",
        "author": "Risk Team",
        "license": "MIT",
        "version": "1.0"
    },
    "parameter_sources": {
        "static_config": {
            "type": "static",
            "values": {
                "eg3config": {
                    "multiplier": 2,
                    "constant": 5.0
                },
            }
        }
    },

    "modules": {
        "example_analysis": {
            "type": "eg1",
            "input_mapping": {
                "spend": "monthly_spend",
                "signups": "new_customers"
            }
        },
        "simple_chain": {
            "type": "eg2",
            "input_mapping": {
                "i1": "input_data_1",
                "i2": "input_data_2"
            }
        },
        "config_example": {
            "type": "eg3",
            "config": "${ped.param:static_config,eg3config}",
            "input_mapping": {
                "i1": "base_value",
                "i2": "adjustment_value"
            }
        }
    }
}

In [6]:
from testing_modules.eg1.config import EG1Module
from testing_modules.eg2.config import EG2Module
from testing_modules.eg3.config import EG3Module

from ped.modules import register_graph_module
register_graph_module(EG1Module)
register_graph_module(EG2Module)
register_graph_module(EG3Module)

In [7]:
flow_config = FlowConfiguration.model_validate(test_config)

parameterized_modules = await flow_config.get_parameterized_modules(inputs=sample_request)
parameterized_modules

ConstructedGraphModules(root={'example_analysis': GraphModule(root=EG1Module(type='eg1', input_mapping={'spend': 'monthly_spend', 'signups': 'new_customers'})), 'simple_chain': GraphModule(root=EG2Module(type='eg2', input_mapping={'i1': 'input_data_1', 'i2': 'input_data_2'})), 'config_example': GraphModule(root=EG3Module(type='eg3', input_mapping={'i1': 'base_value', 'i2': 'adjustment_value'}, config=Eg3Config(multiplier=2, constant=5.0)))})

In [8]:
await flow_config.get_graph(inputs=sample_request)

TypeError: HamiltonGraph.__init__() missing 1 required positional argument: 'default_outputs'